# 00. 可观测性与 Tracing

Agent 的可观测性不同于传统微服务：你需要追踪的不是 HTTP 请求，而是 **Thought → Tool Call → Observation** 的推理链。

Span-based tracing 是标准方案：每个操作（LLM 调用、工具执行、状态更新）都生成一个 span，span 之间有父子关系。

> **检查点**：能画出 Agent 的一次执行对应的 span tree。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# Span-based tracing
from dataclasses import dataclass, field
import time, json

@dataclass
class Span:
    trace_id: str; span_id: str; parent_id: str | None
    name: str; start_ms: float
    end_ms: float = 0; attrs: dict = field(default_factory=dict)
    def finish(self, **attrs):
        self.end_ms = time.time()*1000; self.attrs.update(attrs)

class Tracer:
    def __init__(self, trace_id: str):
        self.trace_id = trace_id; self.spans = []; self._n = 0
    def start(self, name: str, parent_id: str | None = None) -> Span:
        self._n += 1
        s = Span(self.trace_id, f"s{self._n}", parent_id, name, time.time()*1000)
        self.spans.append(s); return s
    def to_dicts(self) -> list[dict]:
        return [{"name":s.name,"parent":s.parent_id,"span":s.span_id,
                 "dur_ms":s.end_ms-s.start_ms,"attrs":s.attrs} for s in self.spans]

# 模拟一次 Agent 执行
tracer = Tracer("trace-001")
root = tracer.start("agent.run")
thought = tracer.start("agent.thought", root.span_id)
thought.finish(content="Analyze user request")
tool = tracer.start("tool.call", root.span_id)
tool.finish(tool="search", args="python", success=True)
result = tracer.start("tool.result", tool.span_id)
result.finish(content_len=42, status="ok")
output = tracer.start("agent.output", root.span_id)
output.finish(tokens=150, status="complete")
root.finish(total_tokens=350)

print("Trace:")
for s in tracer.to_dicts():
    indent = "  " if s["parent"] else ""
    print(f"{indent}[{s['span']}] {s['name']} ({s['dur_ms']:.0f}ms) -> parent={s['parent']}")

Trace:
[s1] agent.run (0ms) -> parent=None
  [s2] agent.thought (0ms) -> parent=s1
  [s3] tool.call (0ms) -> parent=s1
  [s4] tool.result (0ms) -> parent=s3
  [s5] agent.output (0ms) -> parent=s1


## 生产级 Observability 工具

| 工具 | 特点 | 适用场景 |
|------|------|----------|
| LangSmith | LangChain 原生集成 | LangChain/LangGraph 项目 |
| LangFuse | Open Source, OTel | 框架无关，自托管 |
| Arize Phoenix | OpenInference 标准 | 需要 OpenTelemetry 兼容 |
| W&B Weave | 实验跟踪强 | ML-heavy Agent 项目 |

> **检查点**：能说出 LangFuse 和 Arize Phoenix 的核心差异。

## 真实 API 实验

真实 LLM 调用生成 trace：token 数与延迟来自真实 API 响应（`trace_demo.py --use-api`）。


In [3]:
SCRIPT = ROOT / 'agents' / 'production' / 'scripts' / 'trace_demo.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--scenario', 'research_agent', '--use-api', '--output', 'console'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


[trace] span_1: agent.run started
  [trace] span_2: agent.thought (1530ms)
  [trace] span_3: tool.call (0ms)
  [trace] span_4: agent.output (real tokens: 114)
[trace] span_1: complete (3442ms)

Trace summary: 4 spans, 6884ms total duration



## 练习

1. 给 Tracer 增加 error span：当 Agent 步骤失败时，自动创建 error span 并关联到父 span。
2. 实现 trace 导出：将 spans 导出为 OpenTelemetry 兼容的 JSON 格式。
3. 设计一个 Agent 的 dashboard：展示哪些 metrics？（延迟分布、工具调用频率、错误率、成本）